In [1]:
from google.colab import files
uploaded = files.upload()

Saving fallacy-project.zip to fallacy-project.zip


In [3]:
!unzip fallacy-project.zip

Archive:  fallacy-project.zip
   creating: data/
   creating: data/generator_data/
  inflating: data/generator_data/train_generator.jsonl  
  inflating: data/generator_data/val_generator.jsonl  
   creating: data/processed/
  inflating: data/processed/label_map.json  
  inflating: data/processed/test.csv  
  inflating: data/processed/train.csv  
  inflating: data/processed/val.csv  
   creating: data/raw/
  inflating: data/raw/climate_train.csv  
  inflating: data/raw/edu_train.csv  
  inflating: data/raw/esnli_train.csv  
  inflating: data/raw/kialo_flat_clean.csv  
  inflating: data/raw/mappings.csv   
   creating: scripts/
  inflating: scripts/00_setup_colab.py  
  inflating: scripts/1_preprocess_data.py  
  inflating: scripts/2_build_faiss_index.py  
  inflating: scripts/3_train_classifier.py  
  inflating: scripts/4_train_generator.py  
  inflating: scripts/5_cpace_module.py  
  inflating: scripts/6_complete_pipeline.py  
  inflating: scripts/7_training_launcher.py  
   creating: 

In [4]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.0 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [10]:
# ============================================================
# CELL 1: Fast Parallel Packages (No condacolab)
# Run this first - takes 3-4 minutes
# ============================================================

import subprocess
import sys
import torch

print("="*60)
print("INSTALLING PARALLEL PACKAGES (NON-CONDA METHOD)")
print("="*60)

# Install base packages first
base_packages = [
    "torch>=2.0.0",
    "transformers>=4.35.0",
    "sentence-transformers>=2.2.0",
    "scikit-learn>=1.3.0",
    "pandas>=2.0.0",
    "numpy>=1.24.0",
    "tqdm",
    "spacy",
    "accelerate>=0.24.0",
    "joblib>=1.3.0",
    "multiprocess>=0.70.15",
    "numba>=0.59.0",
]

print("\n📦 Installing base packages...")
for package in base_packages:
    print(f"   {package}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", package])

# Install FAISS CPU (works perfectly for 418 passages)
print("\n📦 Installing FAISS...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"])

# Install Ray (distributed framework)
print("\n📦 Installing Ray (this takes ~1 minute)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ray[default]==2.9.0"])

# Install Dask (parallel computing)
print("\n📦 Installing Dask...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "dask[complete]==2024.4.1"])

# Download spaCy model
print("\n📚 Downloading spaCy model...")
subprocess.run(["python", "-m", "spacy", "download", "en_core_web_lg"])

print("\n" + "="*60)
print("✅ ALL PACKAGES INSTALLED SUCCESSFULLY!")
print("="*60)
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Verify installations
print("\n📊 Verification:")
try:
    import ray
    print(f"   ✅ Ray {ray.__version__}")
except Exception as e:
    print(f"   ⚠️ Ray: {e}")

try:
    import dask
    print(f"   ✅ Dask {dask.__version__}")
except Exception as e:
    print(f"   ⚠️ Dask: {e}")

try:
    import faiss
    print(f"   ✅ FAISS {faiss.__version__}")
except Exception as e:
    print(f"   ⚠️ FAISS: {e}")

try:
    import numba
    print(f"   ✅ Numba {numba.__version__}")
except Exception as e:
    print(f"   ⚠️ Numba: {e}")

print("\n🎉 Ready for parallel processing!")

INSTALLING PARALLEL PACKAGES (NON-CONDA METHOD)

📦 Installing base packages...
   torch>=2.0.0
   transformers>=4.35.0
   sentence-transformers>=2.2.0
   scikit-learn>=1.3.0
   pandas>=2.0.0
   numpy>=1.24.0
   tqdm
   spacy
   accelerate>=0.24.0
   joblib>=1.3.0
   multiprocess>=0.70.15
   numba>=0.59.0

📦 Installing FAISS...

📦 Installing Ray (this takes ~1 minute)...

📦 Installing Dask...

📚 Downloading spaCy model...

✅ ALL PACKAGES INSTALLED SUCCESSFULLY!
GPU Available: True
GPU: Tesla T4
Memory: 15.6 GB

📊 Verification:
   ✅ Ray 2.55.1
   ✅ Dask 2024.4.1
   ✅ FAISS 1.13.2
   ✅ Numba 0.60.0

🎉 Ready for parallel processing!


In [8]:
# ============================================================
# CELL 2: Test Parallel Computing
# ============================================================

import time
import numpy as np
from joblib import Parallel, delayed
import torch

print("="*60)
print("TESTING PARALLEL COMPUTING CAPABILITIES")
print("="*60)

# Test 1: PyTorch GPU Parallel
print("\n1️⃣ PyTorch GPU Parallel:")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    # Matrix multiplication on GPU (parallel across thousands of cores)
    a = torch.randn(10000, 10000).to(device)
    b = torch.randn(10000, 10000).to(device)

    start = time.time()
    c = torch.mm(a, b)
    torch.cuda.synchronize()
    elapsed = time.time() - start
    print(f"   ✅ 10,000x10,000 matrix multiplication: {elapsed:.2f}s (GPU parallel)")
else:
    print("   ⚠️ No GPU available")

# Test 2: Joblib CPU Parallel
print("\n2️⃣ Joblib CPU Parallel:")
def square(x):
    return x * x

start = time.time()
results = Parallel(n_jobs=-1)(delayed(square)(i) for i in range(10000))
elapsed = time.time() - start
print(f"   ✅ 10,000 operations across CPU cores: {elapsed:.2f}s")

# Test 3: Numba Parallel
print("\n3️⃣ Numba JIT Parallel:")
from numba import jit, prange
import numpy as np

@jit(nopython=True, parallel=True)
def parallel_sum(arr):
    total = 0.0
    for i in prange(len(arr)):
        total += arr[i]
    return total

arr = np.random.randn(10000000)
start = time.time()
result = parallel_sum(arr)
elapsed = time.time() - start
print(f"   ✅ Sum of 10M elements: {elapsed:.2f}s (parallel)")

print("\n" + "="*60)
print("✅ ALL PARALLEL TESTS PASSED!")
print("="*60)

TESTING PARALLEL COMPUTING CAPABILITIES

1️⃣ PyTorch GPU Parallel:
   ✅ 10,000x10,000 matrix multiplication: 0.69s (GPU parallel)

2️⃣ Joblib CPU Parallel:


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/backend/resource_tracker.py:146: UserWarning: resource_tracker: process died unexpectedly, relaunching. Some folders/semaphores might leak.
  warnings.warn(


   ✅ 10,000 operations across CPU cores: 0.48s

3️⃣ Numba JIT Parallel:
   ✅ Sum of 10M elements: 1.27s (parallel)

✅ ALL PARALLEL TESTS PASSED!


In [28]:
!python /content/scripts/00_setup_colab.py

QUICK WORKING SETUP (2 minutes)
Installing torch...
Installing transformers...
Installing sentence-transformers...
Installing scikit-learn...
Installing pandas...
Installing numpy...
Installing tqdm...
Installing spacy...
Installing accelerate...
Installing faiss-cpu...
Installing ray[default]...
Installing dask[complete]...
Installing joblib...
Installing numba...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 44.6 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')

✅ All packages installed!
ℹ️ Using FAISS CPU (StandardGpuResources not needed)
   This works perfectly for 418 passages

✅ All imports successful!
FAISS version: 1.13.2


In [18]:
!python /content/scripts/1_preprocess_data.py

PARALLEL PREPROCESSING PIPELINE (FULLY FIXED)

🔧 Checking dependencies...
📦 Installing scikit-learn>=1.3.0...

✅ All dependencies ready!

📊 System Info:
   CPU Cores: 2
   Workers: 2
   GPU Available: True
   GPU: Tesla T4

LOADING DATASETS
   ✅ Loaded: climate_train.csv - (855, 4)
   ✅ Loaded: edu_train.csv - (1849, 12)

📊 Combined dataset: 2704 examples
   After cleaning: 2357 examples

📊 Label distribution:
   faulty_generalization: 353
   ad_hominem: 259
   intentional: 259
   false_cause: 199
   appeal_to_emotion: 193
   appeal_to_popularity: 186
   fallacy_of_credibility: 179
   relevance_fallacy: 154
   logical_fallacy: 151
   circular_reasoning: 128
   false_dilemma: 123
   straw_man: 122
   equivocation: 51

APPLYING PARALLEL NER NORMALIZATION
   Using multiprocessing with 2 workers...
Parallel NER: 100% 2357/2357 [00:22<00:00, 102.97it/s]
   ✅ Normalized 2357 texts

CREATING TRAIN/VAL/TEST SPLITS
Total classes: 13

Split sizes:
   Train: 1885
   Val: 236
   Test: 236

✅ Split

In [19]:
# ============================================================================
# DIAGNOSTIC SCRIPT - Check for Data Leakage
# ============================================================================

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

BASE_DIR = "/content"
DATA_PROCESSED = f"{BASE_DIR}/data/processed"

print("="*60)
print("DATA LEAKAGE DIAGNOSTIC")
print("="*60)

# Load splits
train_df = pd.read_csv(f"{DATA_PROCESSED}/train.csv")
val_df = pd.read_csv(f"{DATA_PROCESSED}/val.csv")
test_df = pd.read_csv(f"{DATA_PROCESSED}/test.csv")

print(f"\nSplit sizes:")
print(f"  Train: {len(train_df)}")
print(f"  Val: {len(val_df)}")
print(f"  Test: {len(test_df)}")

# Check for exact duplicates between splits
print("\n🔍 Checking for exact duplicates...")

train_texts = set(train_df['text'].tolist())
val_texts = set(val_df['text'].tolist())
test_texts = set(test_df['text'].tolist())

val_duplicates = len(train_texts.intersection(val_texts))
test_duplicates = len(train_texts.intersection(test_texts))

print(f"  Train-Val duplicates: {val_duplicates}")
print(f"  Train-Test duplicates: {test_duplicates}")

if val_duplicates > 0 or test_duplicates > 0:
    print("  ⚠️ WARNING: Data leakage detected! Same texts in different splits!")

# Check label distribution
print("\n📊 Label distribution in test set:")
label_counts = test_df['label'].value_counts()
for label, count in label_counts.items():
    print(f"  {label}: {count}")

# Check if test labels are balanced
print(f"\n  Total test samples: {len(test_df)}")
print(f"  Unique labels in test: {test_df['label'].nunique()}")

# Check for very short texts (potential memorization)
print("\n📏 Text length analysis:")
for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    lengths = df['text'].str.len()
    print(f"  {name} - Mean: {lengths.mean():.1f}, Min: {lengths.min()}, Max: {lengths.max()}")

# Check for mask token frequency (potential shortcut learning)
print("\n🔖 Mask token [MSK] frequency:")
for name, df in [('Train', train_df), ('Test', test_df)]:
    mask_count = df['text'].str.contains(r'\[MSK\d+\]').sum()
    print(f"  {name}: {mask_count}/{len(df)} ({mask_count/len(df)*100:.1f}%) contain masks")

# Compute similarity between train and test
print("\n🔍 Computing semantic similarity between train and test...")

# Sample to avoid memory issues
sample_train = train_df['text'].sample(min(500, len(train_df))).tolist()
sample_test = test_df['text'].tolist()[:100]

vectorizer = TfidfVectorizer(max_features=1000)
all_texts = sample_train + sample_test
vectors = vectorizer.fit_transform(all_texts)

train_vectors = vectors[:len(sample_train)]
test_vectors = vectors[len(sample_train):]

# Find most similar pairs
similarities = cosine_similarity(test_vectors, train_vectors)
max_similarities = similarities.max(axis=1)

high_similarity = sum(max_similarities > 0.85)
print(f"  Test samples with >85% similarity to training: {high_similarity}/{len(sample_test)}")

if high_similarity > 20:
    print("  ⚠️ WARNING: Test set too similar to training set!")

print("\n" + "="*60)
print("RECOMMENDATIONS:")
print("="*60)
print("1. If duplicates found: Re-split data with stratification AND deduplication")
print("2. If mask tokens are everywhere: Your structure distillation is over-applied")
print("3. If similarities are high: Your test set is not representative")
print("4. Consider using a completely separate dataset for final evaluation")

DATA LEAKAGE DIAGNOSTIC

Split sizes:
  Train: 1885
  Val: 236
  Test: 236

🔍 Checking for exact duplicates...
  Train-Val duplicates: 0
  Train-Test duplicates: 1
  ⚠️ WARNING: Data leakage detected! Same texts in different splits!

📊 Label distribution in test set:
  faulty_generalization: 35
  ad_hominem: 26
  intentional: 26
  false_cause: 20
  appeal_to_emotion: 19
  appeal_to_popularity: 19
  fallacy_of_credibility: 18
  relevance_fallacy: 16
  logical_fallacy: 15
  circular_reasoning: 13
  false_dilemma: 12
  straw_man: 12
  equivocation: 5

  Total test samples: 236
  Unique labels in test: 13

📏 Text length analysis:
  Train - Mean: 150.1, Min: 40, Max: 972
  Val - Mean: 144.5, Min: 41, Max: 595
  Test - Mean: 139.5, Min: 39, Max: 614

🔖 Mask token [MSK] frequency:
  Train: 0/1885 (0.0%) contain masks
  Test: 0/236 (0.0%) contain masks

🔍 Computing semantic similarity between train and test...
  Test samples with >85% similarity to training: 5/100

RECOMMENDATIONS:
1. If dupli

In [ ]:
# ============================================================================
# ALTERNATIVE: Install FAISS GPU via Conda-Forge (More Reliable)
# Run this as a separate cell
# ============================================================================

import subprocess

print("Installing FAISS GPU via conda-forge...")

# Install condacolab
subprocess.run(["pip", "install", "condacolab", "-q"])

# Create and run install script
install_script = """
import condacolab
condacolab.install()

import subprocess
# Install from conda-forge
subprocess.run(["conda", "install", "-y", "-c", "conda-forge", "faiss-gpu", "libfaiss", "libfaiss-avx2"])
print("FAISS GPU installed!")
"""
with open('/tmp/install.py', 'w') as f:
    f.write(install_script)

subprocess.run(["python", "/tmp/install.py"])

# Verify
import faiss
print(f"FAISS version: {faiss.__version__}")
print(f"Has StandardGpuResources: {hasattr(faiss, 'StandardGpuResources')}")

In [29]:
!python /content/scripts/2_build_faiss_index.py

FAISS INDEX BUILDER (CPU/GPU Compatible)
FAISS GPU Support: NO (using CPU)
Loaded 418 passages
Device: cuda
Loading weights: 100% 391/391 [00:00<00:00, 1152.56it/s]
Batches: 100% 7/7 [00:01<00:00,  4.56it/s]
Embeddings shape: (418, 1024)
Building index (dim=1024)...
✅ Index saved to /content/index/faiss/corpus.index

Test search results:
   1. Score: 0.7994 - Islamic Reformed Epistemology's premise that religious belief can be rational wi...
   2. Score: 0.7920 - The democratic method is epistemologically unreliable at any practical problem s...
   3. Score: 0.7884 - Rahul Gandhi is not mature enough to handle a position of responsibility like th...
   4. Score: 0.7878 - En contra de lo que se establece legalmente, no se debería permitir trabajar a m...
   5. Score: 0.7836 - Jesus is not the Messiah of the ancient Hebrew religion....


In [32]:
# Run this to rename files to match the expected names
import os

data_dir = "/content/data/processed"

# Check what files exist
print("Existing files:", os.listdir(data_dir))

# Rename if needed
if os.path.exists(f"{data_dir}/train.csv") and not os.path.exists(f"{data_dir}/train_clean.csv"):
    os.rename(f"{data_dir}/train.csv", f"{data_dir}/train_clean.csv")
    print("✅ Renamed train.csv → train_clean.csv")

if os.path.exists(f"{data_dir}/val.csv") and not os.path.exists(f"{data_dir}/val_clean.csv"):
    os.rename(f"{data_dir}/val.csv", f"{data_dir}/val_clean.csv")
    print("✅ Renamed val.csv → val_clean.csv")

if os.path.exists(f"{data_dir}/test.csv") and not os.path.exists(f"{data_dir}/test_clean.csv"):
    os.rename(f"{data_dir}/test.csv", f"{data_dir}/test_clean.csv")
    print("✅ Renamed test.csv → test_clean.csv")

print("\nFiles now available:", os.listdir(data_dir))

Existing files: ['test.csv', 'train.csv', 'label_map.json', 'val.csv']
✅ Renamed train.csv → train_clean.csv
✅ Renamed val.csv → val_clean.csv
✅ Renamed test.csv → test_clean.csv

Files now available: ['val_clean.csv', 'test_clean.csv', 'train_clean.csv', 'label_map.json']


In [34]:
!python /content/scripts/3_train_classifier.py


DISTRIBUTED TRAINING CONFIGURATION
World Size: 1 GPUs
Local Rank: 0
Device: cuda:0
Mixed Precision: Enabled
CUDA Memory: 15.6 GB

📂 Loading data...
Train: 1885 | Val: 236 | Test: 236
Classes: 13

🤖 Initializing model on 1 GPUs...
Loading weights: 100% 100/100 [00:00<00:00, 16630.86it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because mi

In [36]:
# Run this cell to rename
import os

data_dir = "/content/data/processed"

# Check what files exist
print("Existing files:", os.listdir(data_dir))

# Rename test file
if os.path.exists(f"{data_dir}/test_clean.csv"):
    os.rename(f"{data_dir}/test_clean.csv", f"{data_dir}/test.csv")
    print("✅ Renamed test_clean.csv → test.csv")

# Also rename train and val if needed (generator only needs test)
if os.path.exists(f"{data_dir}/train_clean.csv") and not os.path.exists(f"{data_dir}/train.csv"):
    os.rename(f"{data_dir}/train_clean.csv", f"{data_dir}/train.csv")
if os.path.exists(f"{data_dir}/val_clean.csv") and not os.path.exists(f"{data_dir}/val.csv"):
    os.rename(f"{data_dir}/val_clean.csv", f"{data_dir}/val.csv")

print("\n✅ Files ready!")
print("Now run: !python /content/scripts/4_train_generator.py")

Existing files: ['val_clean.csv', 'test_clean.csv', 'train_clean.csv', 'label_map.json']
✅ Renamed test_clean.csv → test.csv

✅ Files ready!
Now run: !python /content/scripts/4_train_generator.py


In [37]:
!python /content/scripts/4_train_generator.py

SIMPLE EXPLANATION GENERATOR (Using Trained ELECTRA)
Device: cuda

📂 Loading your trained ELECTRA classifier...
Loading weights: 100% 104/104 [00:00<00:00, 7903.17it/s]
Loaded classifier with 13 classes
Classes: ['ad_hominem', 'appeal_to_emotion', 'appeal_to_popularity', 'circular_reasoning', 'equivocation', 'fallacy_of_credibility', 'false_cause', 'false_dilemma', 'faulty_generalization', 'intentional', 'logical_fallacy', 'relevance_fallacy', 'straw_man']

📂 Loading test data...
Test examples: 236

🚀 Generating explanations for test set...
  0% 0/236 [00:00<?, ?it/s]
📝 Text: [PERSON] : msk should oppose any attempt to msk msk . such regulation is the first step to msk of all msk and the elimination of msk constitutional ri...
🎯 Predicted: straw_man (conf: 0.494)
💡 Explanation: This argument commits the STRAW MAN fallacy. It misrepresents the original position on the issue as an exaggerated version, then attacks this distorted version. (Low confidence)

Example: 'You want to...
  0% 1/

In [40]:
# Change en_core_web_lg to en_core_web_sm
!sed -i 's/en_core_web_lg/en_core_web_sm/g' /content/scripts/5_cpace_module.py

print("✅ Script patched to use small model!")
print("Now run: !python /content/scripts/5_cpace_module.py")

✅ Script patched to use small model!
Now run: !python /content/scripts/5_cpace_module.py


In [41]:
!python /content/scripts/5_cpace_module.py

CPACE MODULE TEST
Loading spaCy model for CPACE...

Text: You can't trust Dr. Smith's research because he drives an SUV!
Fallacy: ad_hominem
Alternatives: ['straw_man', 'appeal_to_emotion']

CPACE Explanation:
This argument commits the Ad Hominem fallacy rather than a straw man fallacy. Instead of addressing the logical content about You, it attacks the character or personal attributes of Smith, which is irrelevant to whether the claim is true.

Extracted concepts: {'persons': ['Smith'], 'organizations': [], 'locations': [], 'dates': [], 'clauses': ["You can't trust Dr. Smith's research because he drives an SUV!"], 'topics': ['You', "Dr. Smith's research"], 'count': 1}

Text: Everyone is buying this product, so it must be the best.
Fallacy: appeal_to_popularity
Alternatives: ['false_cause', 'ad_hominem']

CPACE Explanation:
This argument commits the Appeal To Popularity fallacy rather than a false cause fallacy. It assumes that because many people believe or do something, it must be tr

In [45]:
# Install all missing packages for the complete pipeline
!pip install fastapi uvicorn aioredis celery redis -q

print("✅ All web dependencies installed!")
print("Now run: !python /content/scripts/6_complete_pipeline.py")

✅ All web dependencies installed!
Now run: !python /content/scripts/6_complete_pipeline.py


In [47]:
# Replace aioredis with redis.asyncio in the script
!sed -i 's/import aioredis/import redis.asyncio as aioredis/g' /content/scripts/6_complete_pipeline.py

print("✅ Fixed aioredis import!")
print("Now run: !python /content/scripts/6_complete_pipeline.py")

✅ Fixed aioredis import!
Now run: !python /content/scripts/6_complete_pipeline.py


In [49]:
# Modify the script to properly handle asyncio
!sed -i 's/self.batch_task = asyncio.create_task(self._batch_processor())/# self.batch_task = asyncio.create_task(self._batch_processor())/g' /content/scripts/6_complete_pipeline.py
!sed -i 's/self.flush_task = asyncio.create_task(self._periodic_flush())/# self.flush_task = asyncio.create_task(self._periodic_flush())/g' /content/scripts/6_complete_pipeline.py

print("✅ Fixed asyncio issues!")

✅ Fixed asyncio issues!


In [52]:
!python /content/scripts/6_complete_pipeline.py

PARALLEL INFERENCE PIPELINE (CLEAN VERSION)
CPU Cores: 2
GPUs: 1
Batch Size: 32
Max Workers: 2
FALLACY DETECTION PIPELINE (Parallel Version)

🔧 Initializing Parallel Pipeline...
   Device: cuda
   Loading model...
Loading weights: 100% 104/104 [00:00<00:00, 16161.22it/s]
   ✅ Loaded 13 fallacy types
✅ Pipeline ready!


🔬 BENCHMARKING PARALLEL METHODS
/content/scripts/6_complete_pipeline.py:209: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():

📊 Sequential:
   Time: 0.47s
   Throughput: 14.9 texts/sec
   Speedup: 1.00x
/content/scripts/6_complete_pipeline.py:189: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():

📊 GPU Batch:
   Time: 0.12s
   Throughput: 60.3 texts/sec
   Speedup: 4.04x
Parallel inference: 100% 7/7 [00:00<00:00, 110.78it/s]

📊 Thread Parallel:
   Time: 0.07